# Medical Screening with Online Conformal Prediction

A pathology lab processes tissue samples one at a time. A model helps clinicians
flag malignant cases — but a bare label is rarely enough. Clinicians need to know
*how confident* the model is, how to act on uncertain predictions, whether
different patient cohorts receive equally reliable predictions, and whether
the incoming patient population has started to drift.

This notebook answers six practical questions using
**online conformal prediction** on the Wisconsin Breast Cancer dataset. Data
arrives sample-by-sample; the model updates after every prediction.

| Section | Question | Tool |
|---------|----------|---------|
| 1 | Setup | `numpy`, `sklearn.datasets` |
| 2 | How confident is this prediction? | `Pipeline`, `StandardScaler`, `ConformalSupportVectorMachine` |
| 3 | Give me a probability, not a set | `Pipeline`, `VennAbersPredictor` |
| 4 | What should I actually do? | `venn_decision`, `UtilityFunction` |
| 5 | Are we treating all cohorts fairly? | `MondrianConformalClassifier` |
| 6 | Is the model still working? | `VariationalLegendreJumper`, `VilleWrapper`, `CUSUMWrapper` |

All data is offline (no network required). No theory prerequisites — see the
[quickstart](https://egonmedhatten.github.io/online-cp/quickstart/) and
[guide](https://egonmedhatten.github.io/online-cp/guide/) for background.

## 1. Setup

Load the Wisconsin Breast Cancer dataset (569 samples, 30 features).  
Labels: `0` = malignant, `1` = benign.

We keep the raw (unscaled) data here and let `Pipeline` handle standardisation
in each subsequent section — that way preprocessing and prediction stay together
as a single object with a common `learn_initial_training_set` / `learn_one` /
`predict` interface.

The deployment stream is split into:
- **280 stable samples** drawn from the same distribution as training
- **89 drifted samples** — mean radius and texture shifted by +1.5 SD to
  simulate a new referral protocol arriving at the lab.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 110, "figure.figsize": (8, 3)})

from sklearn.datasets import load_breast_cancer

# ── Load raw data (no external scaler) ───────────────────────────────────────
bc = load_breast_cancer()
rng = np.random.default_rng(42)
idx = rng.permutation(len(bc.target))
X_raw = bc.data[idx].astype(float)
y_all = bc.target[idx]   # 0 = malignant, 1 = benign

# ── Splits ────────────────────────────────────────────────────────────────────
N_TRAIN  = 200   # initial training set
N_STABLE = 280   # stable deployment stream
# remaining 89 samples form the drifted tail

X_train_raw, y_train = X_raw[:N_TRAIN], y_all[:N_TRAIN]
X_stable_raw, y_stable = X_raw[N_TRAIN:N_TRAIN + N_STABLE], y_all[N_TRAIN:N_TRAIN + N_STABLE]

# Inject drift: shift mean radius (col 0) and mean texture (col 1)
# Amount = 1.5 * training SD of each feature
train_std = X_train_raw.std(axis=0)
X_drifted_raw = X_raw[N_TRAIN + N_STABLE:].copy()
X_drifted_raw[:, 0] += 1.5 * train_std[0]
X_drifted_raw[:, 1] += 1.5 * train_std[1]
y_drifted = y_all[N_TRAIN + N_STABLE:]

# Full deployment stream: stable then drifted
X_deploy_raw = np.vstack([X_stable_raw, X_drifted_raw])
y_deploy     = np.concatenate([y_stable, y_drifted])
DRIFT_START  = N_STABLE   # index within deployment stream where drift begins

print(f"Training:   {N_TRAIN} samples")
print(f"Stable:     {N_STABLE} samples")
print(f"Drifted:    {len(X_drifted_raw)} samples")
print(f"Label dist (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")


## 2. How confident is this prediction?

A plain classifier outputs a single label. A **conformal predictor** outputs a
*prediction set* — often a singleton, but sometimes both labels, sometimes empty.

The size of the prediction set encodes uncertainty in a statistically rigorous
way: the true label is guaranteed to be in the set with probability ≥ 1 − ε
**regardless of the data distribution**, with no calibration assumptions.

We compose `StandardScaler` (frozen on the training batch) with
`ConformalSupportVectorMachine` using the `|` operator. The resulting `Pipeline`
has the same `learn_initial_training_set` / `learn_one` / `predict` interface
as any bare predictor — preprocessing and prediction travel together.

In [ ]:
from online_cp import (
    Pipeline, StandardScaler,
    ConformalSupportVectorMachine,
    plot_coverage, plot_set_sizes,
)
from online_cp.metrics import ErrorRate, SetSize

EPSILON = 0.10  # target error rate

svm_pipe = StandardScaler() | ConformalSupportVectorMachine(
    kernel="rbf", sigma=1.0, C=5.0,
    nonconformity="margin",
    label_space=np.array([0, 1]),
    epsilon=EPSILON,
    rnd_state=0,
)
svm_pipe.learn_initial_training_set(X_train_raw, y_train)

err_rate = ErrorRate()
set_size = SetSize()
prediction_sets = []
p_values_stream = []   # p-value of the true label — used in Section 6

for x, y in zip(X_deploy_raw, y_deploy):
    Gamma, pv = svm_pipe.predict(x, epsilon=EPSILON, return_p_values=True)
    err_rate.update(y=y, Gamma=Gamma)
    set_size.update(y=y, Gamma=Gamma)
    prediction_sets.append(Gamma)
    p_values_stream.append(pv[y])   # uniform under H0 (exchangeability)
    svm_pipe.learn_one(x, y)

print(f"Empirical coverage: {1 - np.mean(err_rate.values):.3f}  (nominal: {1-EPSILON:.2f})")
print(f"Avg set size:       {np.mean(set_size.values):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_coverage(err_rate, nominal=1-EPSILON, ax=axes[0], color="steelblue")
axes[0].set_title("Running coverage")
plot_set_sizes(set_size, ax=axes[1], color="steelblue")
axes[1].axvline(DRIFT_START, color="red", linestyle=":", label="Drift starts")
axes[1].legend()
axes[1].set_title("Running avg set size")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect individual predictions — set size as a practical uncertainty signal
print(f"{'Step':>5}  {'True label':>12}  {'Prediction set':>18}  {'Interpretation'}")
print("-" * 68)
interp = {
    frozenset([0]):    "Confident: malignant",
    frozenset([1]):    "Confident: benign",
    frozenset([0, 1]): "Uncertain — flag for review",
    frozenset():       "Anomaly — neither label fits",
}
for i in range(20):
    G = prediction_sets[i]
    lbl = "malignant" if y_deploy[i] == 0 else "benign"
    desc = interp.get(frozenset(G.elements.tolist()), "?")
    print(f"{i:>5}  {lbl:>12}  {str(sorted(G.elements.tolist())):>18}  {desc}")


## 3. Give me a probability, not a set

Clinicians often prefer a calibrated probability to a set. The `VennAbersPredictor`
produces a **multiprobability** — an interval $[p_0, p_1]$ rather than a point.

- $p_0$ = lower bound on $P(y=\text{benign})$ (assuming the true label is malignant)
- $p_1$ = upper bound on $P(y=\text{benign})$ (assuming the true label is benign)

The midpoint $(p_0+p_1)/2$ is a theoretically calibrated probability with no
distributional assumptions. The interval width is a second-order signal:
wide = the model is uncertain even about its own probability.

Again we use `Pipeline` so scaling is part of the predictor object itself.

In [ ]:
from online_cp import VennAbersPredictor, plot_reliability_diagram_venn

vap_pipe = StandardScaler() | VennAbersPredictor(
    scorer="svm", kernel="rbf", sigma=1.0, C=5.0,
)
vap_pipe.learn_initial_training_set(X_train_raw, y_train)

venn_preds  = []
venn_labels = []

for x, y in zip(X_deploy_raw, y_deploy):
    venn_preds.append(vap_pipe.predict(x))
    venn_labels.append(y)
    vap_pipe.learn_one(x, y)

print(f"{'Step':>5}  {'True':>8}  {'p0':>6}  {'p1':>6}  {'midpoint':>10}  {'width':>8}  {'call'}")
print("-" * 68)
for i in range(20):
    p   = venn_preds[i]
    mid = (p.p0 + p.p1) / 2
    call = "benign" if mid > 0.5 else "malignant"
    lbl  = "benign" if venn_labels[i] == 1 else "malignant"
    print(f"{i:>5}  {lbl:>8}  {p.p0:>6.3f}  {p.p1:>6.3f}  {mid:>10.3f}  {p.p1 - p.p0:>8.3f}  {call}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_reliability_diagram_venn(
    venn_preds, venn_labels,
    which="both", n_bins=12, target_label=1, ax=ax,
)
ax.set_title("Reliability: Venn-Abers (hypothesis curve = calibrated)")
plt.tight_layout()
plt.show()


## 4. What should I actually do?

A screening model is only useful if it informs a decision. Different decisions
have asymmetric costs:

| Decision | Missed malignant | Unnecessary biopsy |
|----------|-----------------|-------------------|
| Cost | Very high | Moderate |

We model this with a `UtilityFunction` that penalises false negatives 8× more
than false positives. The optimal decision at each step maximises *expected
utility* under the Venn-Abers probability estimates — automatically shifting
the effective decision threshold to account for the cost asymmetry.

In [ ]:
from online_cp import UtilityFunction, venn_decision

# Actions: 0 = refer for biopsy, 1 = discharge
# Labels:  0 = malignant,        1 = benign
#
# U[action, true_label]
U = np.array([[ 1, -1],   # refer:     correct detection / unnecessary biopsy
              [-8,  1]])  # discharge: dangerous miss    / correct discharge

utility = UtilityFunction(fn=lambda x, y, d: U[d, y], decisions=[0, 1])

decisions       = [venn_decision(pred, utility, x=None) for pred in venn_preds]
naive_decisions = [1 if (p.p0 + p.p1) / 2 > 0.5 else 0 for p in venn_preds]

y_dep = np.array(venn_labels)
d_arr = np.array(decisions)
n_arr = np.array(naive_decisions)

def error_summary(dec, y_true, name):
    fn = np.sum((dec == 1) & (y_true == 0))   # discharged a malignant
    fp = np.sum((dec == 0) & (y_true == 1))   # referred a benign
    print(f"{name:<24}  false negatives (missed malignant) = {fn:>3},  "
          f"false positives (unnecessary referral) = {fp}")

error_summary(d_arr, y_dep, "Utility-optimal")
error_summary(n_arr, y_dep, "Naïve 0.5 threshold")


## 5. Are we treating all cohorts fairly?

Marginal coverage holds on average across all patients — but different subgroups
may see systematically different set sizes and effective coverages. A **Mondrian
conformal predictor** enforces *group-conditional* coverage:

$$P(y \in \Gamma(x) \mid \kappa(x) = k) \geq 1 - \varepsilon \quad \text{for every group } k$$

We partition patients by **mean radius** (feature 0) into low- and high-radius
cohorts. The pre-fitted scaler from the SVM pipeline transforms the raw features
so the Mondrian threshold (0 in standardised units = population median) is
directly meaningful.

In [ ]:
from online_cp import MondrianConformalClassifier

# Reuse the scaler fitted in Section 2 — parameters are identical since
# both pipelines saw the same X_train_raw.
fitted_scaler = svm_pipe.transformers[0]
X_train_sc  = fitted_scaler.transform(X_train_raw)
X_stable_sc = fitted_scaler.transform(X_stable_raw)

# Category function operates on standardised features (threshold 0 = training median)
def cohort_fn(x):
    return "high-radius" if x[0] > 0.0 else "low-radius"

def make_svm():
    return ConformalSupportVectorMachine(
        kernel="rbf", sigma=1.0, C=5.0,
        nonconformity="margin",
        label_space=np.array([0, 1]),
        epsilon=EPSILON, rnd_state=1,
    )

mondrian_svm = MondrianConformalClassifier(base_model=make_svm(), category_fn=cohort_fn)
plain_svm    = make_svm()

mondrian_svm.learn_initial_training_set(X_train_sc, y_train)
plain_svm.learn_initial_training_set(X_train_sc, y_train)

results = {"plain": {"low": [], "high": []}, "mondrian": {"low": [], "high": []}}

for x_sc, y_true in zip(X_stable_sc, y_stable):
    key = "low" if cohort_fn(x_sc) == "low-radius" else "high"
    results["plain"][key].append(int(y_true in plain_svm.predict(x_sc, epsilon=EPSILON)))
    plain_svm.learn_one(x_sc, y_true)
    results["mondrian"][key].append(int(y_true in mondrian_svm.predict(x_sc, epsilon=EPSILON)))
    mondrian_svm.learn_one(x_sc, y_true)

print(f"{'Predictor':<12} {'Low-radius cov':>18} {'High-radius cov':>18} {'Gap':>8}")
print("-" * 60)
for name in ("plain", "mondrian"):
    lo = np.mean(results[name]["low"])
    hi = np.mean(results[name]["high"])
    print(f"{name:<12} {lo:>18.3f} {hi:>18.3f} {abs(lo - hi):>8.3f}")


## 6. Is the model still working?

When a new referral protocol sends patients with different morphology, the
p-values from the conformal predictor cluster near 0 — the new patients look
anomalous to the model. A **conformal test martingale** turns this p-value
stream into a formal sequential test of exchangeability.

We use the `VariationalLegendreJumper` (Algorithm 4, new in v0.3.0) as the
underlying martingale, then wrap it in two detection procedures:

- **`VilleWrapper`** — reject when the running *maximum* of $M_n$ exceeds a
  threshold $c$.  By Ville's inequality $P(\exists n : M_n \geq c) \leq 1/c$,
  so $c=100$ gives a 1% false-alarm guarantee.
- **`CUSUMWrapper`** — tracks the ratio $\gamma_n = M_n / \min_{i\leq n} M_i$
  (Page CUSUM).  Resets the "debt" accumulated before the change-point, giving
  faster detection at the cost of needing a separate threshold calibration.

Both wrappers expose a `plot_detector`-compatible interface.

In [ ]:
from online_cp import (
    VariationalLegendreJumper,
    VilleWrapper, CUSUMWrapper,
    plot_detector,
)

print(f"Total p-values: {len(p_values_stream)}  "
      f"({DRIFT_START} stable + {len(p_values_stream) - DRIFT_START} drifted)")

# Two independent martingales — same algorithm, different wrappers
vlj_ville = VariationalLegendreJumper(orders=[1, 2, 3], J=0.01)
vlj_cusum = VariationalLegendreJumper(orders=[1, 2, 3], J=0.01)

ville = VilleWrapper(vlj_ville, threshold=100)   # 1% significance
cusum = CUSUMWrapper(vlj_cusum)

for p in p_values_stream:
    ville.update(p)
    cusum.update(p)

print(f"\nVille  — rejected: {ville.rejected},  "
      f"alarm time: {ville.rejection_time}")
cusum_alarm = next((i for i, g in enumerate(cusum.cusum_values) if g > 100), None)
print(f"CUSUM  — alarm at threshold 100: step {cusum_alarm}")
if ville.rejection_time is not None:
    print(f"Drift injected at step {DRIFT_START}, "
          f"Ville detected {ville.rejection_time - DRIFT_START} steps later")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

plot_detector(ville, threshold=100, log_scale=True,
              change_point=DRIFT_START, ax=axes[0], color="steelblue")
axes[0].set_title("Ville procedure (running max)")

plot_detector(cusum, threshold=100, log_scale=True,
              change_point=DRIFT_START, ax=axes[1], color="darkorange")
axes[1].set_title("CUSUM procedure (ratio to running min)")

plt.tight_layout()
plt.show()


## Summary

| Question | Answer | Tool |
|----------|--------|------|
| How confident? | Prediction sets; size encodes uncertainty | `Pipeline`, `StandardScaler`, `ConformalSupportVectorMachine` |
| Probability? | Calibrated multiprobability interval $[p_0, p_1]$ | `Pipeline`, `VennAbersPredictor` |
| What to do? | Utility-optimal decisions; asymmetric costs shift the threshold | `venn_decision`, `UtilityFunction` |
| Fair across cohorts? | Mondrian CP equalises coverage per group | `MondrianConformalClassifier` |
| Still working? | Martingale + detection wrappers flag distribution shift | `VariationalLegendreJumper`, `VilleWrapper`, `CUSUMWrapper` |

All five tools share the same **online update protocol** —
`learn_initial_training_set` then `learn_one` after each prediction — and
compose naturally via `Pipeline`.

**Next steps**
- See the [pipeline notebook](pipeline.ipynb) for save/load, `TransformerUnion`,
  and feature-selection transformers.
- See the [decision-making notebook](decision-making.ipynb) for the full
  expected-utility framework including soft decisions and regret analysis.